# gamma=2：固定配对验证

在Colab打开本Notebook后Run all即可；无需手动上传源码ZIP。默认直接下载公开B研究分支dev/local-joint-state-payload-v1中的gamma2入口及可用源码，放入隔离目录。第一代码单元仅挂载Drive。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. 输入与隔离源码

复用旧fixed-four-new-v1的P0终态、原始families及缓存。新输出B-Gamma2-Fixed-Validation-V1/fixed-four-source-v1。已有attempt时，在下载/setup前只读恢复。可选SOURCE_DIRECTORY先复制到隔离目录，原目录不会被覆盖。


In [ ]:

from pathlib import Path
import json, os, shutil, stat, subprocess, sys, tempfile, urllib.request, zipfile
INPUT = Path('/content/drive/MyDrive/Video-WM/B-Terminal-Three-Group-V1/fixed-four-new-v1')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/B-Gamma2-Fixed-Validation-V1/fixed-four-source-v1')
SOURCE_REF = 'dev/local-joint-state-payload-v1'
SOURCE_DIRECTORY = None  # Optional usable local tree; copied into an isolated directory.
HOSTS=(('b2-m3-c01','cal',2),('b2-m3-c02','cal',11),('b2-m3-e01','eval',17),('b2-m3-e02','eval',6))

ARMS=('OFF','P0','PM1','PM2')

KEYS={'key00':'watermark','key01':'watermark-wrong'}

WINDOWS={'L':0,'M':53,'R':106}

FIXTURES={'CROP':(0,None),'DELETE_LEFT_GAP':(8,49),'DELETE_RIGHT_GAP':(8,102)}

def recover_ledger(output):
    """Read-only receipt overlay; also embedded verbatim in the notebook.

    STARTED remains spent and incomplete. Invalid or uncommitted receipts
    remain unresolved, never an inferred successful or unentered call.
    """
    from collections import Counter
    import json
    from pathlib import Path
    output = Path(output)
    ledger={slot:dict(slot=slot,kind=kind,status='NOT_ENTERED',attempts=0) for slot,kind in plan_units().items()}
    errors, receipt_slots = {}, []
    try:
        saved = json.loads((output/'ledger.json').read_text(encoding='utf-8'))
        if not isinstance(saved,dict):raise ValueError('ledger must map slots to records')
        for slot,row in saved.items():
            if slot not in ledger:
                errors['ledger/'+slot]='unknown saved slot';continue
            if not isinstance(row,dict) or row.get('kind')!=ledger[slot]['kind']:
                ledger[slot].update(status='UNRESOLVED_LEDGER',attempts=1 if isinstance(row,dict) and row.get('attempts')==1 else None)
                errors['ledger/'+slot]='invalid saved slot';continue
            status,attempts=row.get('status'),row.get('attempts')
            if status not in ('NOT_ENTERED','STARTED','RETURNED','FAILED','INTERRUPTED') or type(attempts) is not int or attempts!=(0 if status=='NOT_ENTERED' else 1):
                ledger[slot].update(status='UNRESOLVED_LEDGER',attempts=1 if attempts==1 else None);errors['ledger/'+slot]='invalid status/attempt';continue
            ledger[slot]=row
    except (OSError, ValueError) as error:
        errors['ledger.json'] = str(error)
    directory = output/'receipts'
    paths = sorted(directory.rglob('*.json')) + sorted(directory.rglob('*.json.tmp'))
    for path in paths:
        relative = path.relative_to(directory).as_posix()
        pending = relative.endswith('.tmp')
        slot = relative[:-9] if pending else relative[:-5]
        if slot not in ledger:
            errors['receipts/'+relative]='receipt outside fixed plan';continue
        previous = ledger[slot]
        receipt = None
        try:
            receipt = json.loads(path.read_text(encoding='utf-8'))
            if pending:
                raise ValueError('uncommitted receipt; completion is unresolved')
            if not isinstance(receipt, dict) or receipt.get('slot') != slot:
                raise ValueError('receipt slot does not match its address')
            status, attempts = receipt.get('status'), receipt.get('attempts')
            if status not in ('NOT_ENTERED', 'STARTED', 'RETURNED', 'FAILED', 'INTERRUPTED'):
                raise ValueError('invalid receipt status')
            if type(attempts) is not int or attempts != (0 if status == 'NOT_ENTERED' else 1):
                raise ValueError('invalid receipt attempt count')
            if not isinstance(receipt.get('kind'), str) or not receipt['kind']:
                raise ValueError('receipt kind required')
            if previous.get('kind') is not None and previous['kind'] != receipt['kind']:
                raise ValueError('receipt kind disagrees with saved slot')
            if previous.get('attempts') == 1 and attempts == 0:
                raise ValueError('receipt cannot erase a spent call')
            ledger[slot] = receipt
            receipt_slots.append(slot)
        except (OSError, ValueError) as error:
            name = 'receipts/'+relative
            errors[name] = str(error)
            ledger[slot] = dict(previous, slot=slot, status='UNRESOLVED_RECEIPT',
                                attempts=1 if previous.get('attempts') == 1 or isinstance(receipt,dict) and receipt.get('attempts')==1 else None,
                                saved_status=previous.get('status'), recovery_error=name)
    counts = dict(Counter(row.get('status', 'UNRESOLVED_LEDGER') for row in ledger.values()))
    kinds = sorted({row.get('kind', 'UNKNOWN') for row in ledger.values()})
    unit_counts = {kind: dict(Counter(row.get('status', 'UNRESOLVED_LEDGER') for row in ledger.values()
                                     if row.get('kind', 'UNKNOWN') == kind)) for kind in kinds}
    return dict(ledger=ledger, counts=counts, unit_counts=unit_counts,
                receipt_slots=receipt_slots, read_errors=errors, physical_reentry=False,denominators=denominators())

def denominators():
    return dict(sources=4,cal_sources=2,eval_sources=2,arms=4,gamma_candidates=1,
        D181=4,E41=144,codec_encode=16,codec_probe=16,codec_decode=16,E41_frames=5904,
        new_Q=216,new_families=54,new_score_slots=7398,eval_DCT=72,eval_key_decodes=144,
        calibration_unique_families=108,calibration_unique_score_slots=14796,
        eval_tasks=48,aggregate_slots_each=1536,positive_slots=384,null_unique_slots=1152,
        per_gamma_positive=192,per_gamma_null=960,shared_null=768,local_CDC_slots=4608,
        path_classes_each_method=46128,quality_new=36,quality_frames=5796,quality_adjacent=5760,bundles=0)

def execution_artifacts(output):
    from pathlib import Path
    output=Path(output)
    names=('attempt.json','plan.json','ledger.json','inputs.json','background.json','thresholds.json',
           'physical_counts.json','result.json','evaluation.json','planned_outputs.json','blind_complete.json','post_truth_access.json')
    found=[n+s for n in names for s in ('','.tmp') if (output/(n+s)).exists() or (output/(n+s)).is_symlink()]
    for name in ('receipts','private','media','observations','families','blind','quality','calibration'):
        folder=output/name
        if folder.is_dir():found.extend(str(p.relative_to(output)) for p in folder.rglob('*') if p.is_file() or p.is_symlink())
    return sorted(found)

def plan_units():
    units={'inputs':'input_semantics','preflight':'saved_input_preflight','controls':'saved_control_contexts','calibration/PM1':'calibrate90',
           'calibration/PM2':'calibrate90','calibration/common':'freeze_common','blind_complete':'blind_readback','post':'post_evaluation'}
    for h,role,_ in HOSTS:
        for suffix,kind in [('construct','construct_gamma2'),('native','native_audit'),('vae_load','model_load'),('release','model_release')]:units[h+'/'+suffix]=kind
        units[h+'/full/D']='D181';units[h+'/full/preclip']='clipping';units[h+'/full/quantize']='quantize'
        for scope in ('full',*FIXTURES):
            if scope!='full':units[h+'/'+scope+'/edit']='fixture151'
            for suffix,kind in [('encode','codec_encode'),('probe','codec_probe'),('decode','codec_decode')]:units[h+'/'+scope+'/'+suffix]=kind
        for fixture in FIXTURES:
            for window in WINDOWS:
                stem=f'{h}/{fixture}/{window}'
                for phase in range(4):
                    view=stem+f'/p{phase}';units[view+'/E']='E41'
                    for key in (('key01',) if role=='cal' else KEYS):units[view+'/'+key+'/Q']='Q'
                    if role=='eval':
                        units[view+'/DCT']='DCT'
                        for key in KEYS:units[view+'/'+key+'/bits']='key_bits'
                for key in (('key01',) if role=='cal' else KEYS):units[stem+'/'+key+'/score']='H137_score'
        for scope,layers in [('full',('preclip','rgb8','readback'))]+[(f,('rgb8','readback')) for f in FIXTURES]:
            for layer in layers:units[f'{h}/quality/{scope}/{layer}']='quality'
    for h,role,_ in HOSTS:
        if role!='eval':continue
        for arm in ARMS:
            for fixture in FIXTURES:
                for key in KEYS:
                    task='/'.join((h,arm,fixture,key))
                    units['blind/'+task+'/plan']='blind_plan';units['blind/'+task+'/read']='blind_read'
    return units
EXISTING_ARTIFACTS = execution_artifacts(OUTPUT)
SOURCE_ROOT = None
if EXISTING_ARTIFACTS:
    print('已有一次执行记录：仅恢复读取，不下载源码、不安装、不加载模型。')
else:
    workspace=Path(tempfile.mkdtemp(prefix='b-gamma2-source-')).resolve()
    if SOURCE_DIRECTORY is None:
        archive=workspace/'source.zip'
        url='https://codeload.github.com/RICHAAARC/SC-SSTW/zip/refs/heads/'+SOURCE_REF
        urllib.request.urlretrieve(url,archive)
        destination=workspace/'source';destination.mkdir()
        with zipfile.ZipFile(archive) as source:
            for member in source.infolist():
                normalized=member.filename.replace('\\','/')
                target=(destination/normalized).resolve()
                if normalized.startswith('/') or any(':' in p for p in normalized.split('/')) or not target.is_relative_to(destination.resolve()) or stat.S_ISLNK(member.external_attr>>16):
                    raise ValueError('unsafe source archive entry: '+member.filename)
            source.extractall(destination)
        entries=list(destination.glob('*/experiments/wan_state_clock/b_gamma2_fixed_validation_v1_run.py'))
        if len(entries)!=1:raise FileNotFoundError('Gamma2 source entry unavailable: '+url)
        SOURCE_ROOT=entries[0].parents[2]
    else:
        original=Path(SOURCE_DIRECTORY).expanduser().resolve()
        if not (original/'experiments/wan_state_clock/b_gamma2_fixed_validation_v1_run.py').is_file():
            raise FileNotFoundError('Gamma2 entry missing in SOURCE_DIRECTORY')
        SOURCE_ROOT=workspace/'source'
        shutil.copytree(original,SOURCE_ROOT,ignore=shutil.ignore_patterns('.git','__pycache__'))
    OUTPUT.mkdir(parents=True,exist_ok=True)
    print('隔离源码目录:',SOURCE_ROOT)
    print('原始输入:',INPUT,'新输出:',OUTPUT)


## 2. 固定一次预算

仅PM2新增4D181、144E41，codec encode/probe/decode各16；无生成、DiT或E181。校准只用wrong key；双90族最大值得到共同108族阈值。首失败停止，未进入行保留，不补跑。四arm的48项新盲计划全部保存回读后才读bit值，最后开放真值。


In [ ]:
if EXISTING_ARTIFACTS:
    print('只读已有执行记录。')
else:
    LOG = OUTPUT/'setup.log'
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode
    def ensure_imports(probe, packages, label):
        import re
        mapping = {p.replace('-', '_'):p for p in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1).split('.')[0] in mapping:
                package = mapping[missing.group(1).split('.')[0]]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' actual import failure; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, action)
    ensure_imports('import numpy,torch', ['numpy','torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub,transformers,sentencepiece,ftfy; from diffusers import WanPipeline,AutoencoderKLWan',
        ['diffusers','accelerate','safetensors','huggingface-hub','transformers','sentencepiece','ftfy'], 'Wan pipeline and VAE')
    import shutil
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        command(['apt-get','-qq','update'], 'Prepare media dependency')
        command(['apt-get','-qq','install','-y','ffmpeg'], 'Install FFmpeg')
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        raise RuntimeError('ffmpeg/ffprobe unavailable; see '+str(LOG))

    probe="from pathlib import Path; import importlib; root=Path.cwd().resolve(); names=['runtime.wan.b_gamma2_fixed_validation_v1','runtime.wan.b_gamma2_fixed_validation_backend_v1','runtime.wan.b_gamma2_fixed_validation_evaluation_v1','experiments.wan_state_clock.b_gamma2_fixed_validation_v1_run']; modules=[importlib.import_module(n) for n in names]; assert all(Path(m.__file__).resolve().is_relative_to(root) for m in modules); print(*[m.__file__ for m in modules],sep='\\n')"
    subprocess.run([sys.executable,'-c',probe],cwd=SOURCE_ROOT,check=True)
    args=[sys.executable,'-u','-m','experiments.wan_state_clock.b_gamma2_fixed_validation_v1_run',
          '--input',str(INPUT),'--output',str(OUTPUT)]
    with (OUTPUT/'runner.log').open('a') as stream:
        worker=subprocess.Popen(args,cwd=SOURCE_ROOT,stdout=stream,stderr=subprocess.STDOUT)
        try:
            while worker.poll() is None:
                try:worker.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    progress=OUTPUT/'progress.json'
                    try:
                        value=json.loads(progress.read_text());print(value.get('active'),value.get('counts'),flush=True)
                    except (OSError,ValueError):print('同一worker运行中；详见runner.log。',flush=True)
        except BaseException:
            worker.terminate()
            try:worker.wait(timeout=10)
            except subprocess.TimeoutExpired:worker.kill();worker.wait()
            raise
    print('Runner exit:',worker.returncode,'保存目录:',OUTPUT)


## 3. 结果摘要

每gamma192正例、960null；双gamma唯一null1152。主读法CDC_M；实际supporting_refs精确同组支持单独后评。只新增36份P0→PM2质量记录，不加入跨片bundle。完整明细存Drive，不把巨型计划/引用输出到Notebook。


In [ ]:

recovery=recover_ledger(OUTPUT)
print('恢复后的执行状态:',recovery['counts'])
print('读取异常数量:',len(recovery['read_errors']))
for name in ('result.json','thresholds.json','evaluation.json'):
    try:value=json.loads((OUTPUT/name).read_text())
    except (OSError,ValueError) as error:
        print(name,'尚不可读:',type(error).__name__);continue
    if name=='result.json':
        print('状态:',value.get('status'),'固定分母:',value.get('denominators'))
        print('物理调用:',{k:{n:v.get(n) for n in ('planned','started','returned','failed')} for k,v in value.get('physical_counts',{}).items()})
        print('停止原因:',value.get('stop'))
    elif name=='thresholds.json':print('共同校准:',{k:value.get(k) for k in ('status','tau1','tau2','tau','planned_families')})
    else:
        print('开发性结论:',value.get('development_support'),'规则:',value.get('support_checks'))
        print('CDC_M正例:',value.get('totals',{}).get('CDC_M/positive'),'null:',value.get('totals',{}).get('CDC_M/null'))
        for name,pair in value.get('paired',{}).items():
            print(name,'新增实际精确支持正确槽:',pair.get('added_actual_supported_correct'),
                  '丢失:',pair.get('lost_actual_supported_correct'),'共同输出数:',pair.get('common_count'))
print('完整计划、引用、失败明细和质量记录保留在:',OUTPUT)
print('仅固定已见来源的开发性配对证据；未宣称总体FPR、普遍鲁棒性或感知质量PASS。')
